# 01 — Benchmark local

**OBRIGATÓRIO:** Select Kernel → `/Users/wolfx/Documents/Dev/Celx/.venv/bin/python`

Se aparecer `CommandLineTools` no caminho do Python, o kernel está ERRADO.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

EXPECTED = Path('/Users/wolfx/Documents/Dev/Celx/.venv/bin/python').resolve()
current = Path(sys.executable).resolve()
print("Python do kernel:", current)

if current != EXPECTED and ".venv" not in str(current):
    raise RuntimeError(
        "Kernel ERRADO (Python do sistema).\n"
        "Cursor: Select Kernel (canto do notebook)\n"
        f"Escolha: {EXPECTED}\n"
        f"Atual: {current}\n"
        "Depois: Restart Kernel e rode de novo."
    )

ROOT = Path('/Users/wolfx/Documents/Dev/Celx').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ["PYTHONPATH"] = str(ROOT)
print("Kernel OK (.venv)")


## Localizar o projeto

Sobe diretórios até achar `configs/default.yaml`.


In [ ]:
here = Path.cwd().resolve()
repo_dir = None
for candidate in [here, *here.parents]:
    if (candidate / 'configs' / 'default.yaml').exists():
        repo_dir = candidate
        break
assert repo_dir is not None, (
    'Abra o notebook a partir da raiz do Celx ou ajuste o working directory.'
)

os.environ['PYTHONPATH'] = str(repo_dir)
os.chdir(repo_dir)
print('Projeto:', repo_dir)
print('CWD:', Path.cwd())


In [ ]:
# Instala deps SÓ no .venv (nunca no Python da Apple)
pkgs = [
    "transformers>=4.51", "accelerate>=1.0", "datasets>=3.0",
    "PyYAML>=6.0", "pandas>=2.0", "peft>=0.14", "trl>=0.15", "torch",
]
print("Instalando no .venv (pode demorar)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
# NÃO usa pip install -e . — evita Permission denied
import legacy_doc
print("legacy_doc OK:", legacy_doc.__version__)
print("Deps OK")


## Casos do benchmark

Usa `configs/benchmark_local.yaml` + `dataset/benchmark/baseline.jsonl` (4 casos).


In [ ]:
import json
import yaml

benchmark_config = repo_dir / 'configs/benchmark_local.yaml'
benchmark_path = repo_dir / 'dataset/benchmark/baseline.jsonl'
assert benchmark_config.exists(), f'Falta {benchmark_config}'
assert benchmark_path.exists(), f'Falta {benchmark_path}'

records = [
    json.loads(line)
    for line in benchmark_path.read_text(encoding='utf-8').splitlines()
    if line.strip()
]
print(f'{len(records)} casos em {benchmark_path.relative_to(repo_dir)}')
for row in records:
    print('-', row['id'], row['language'])
print('Config:', benchmark_config.relative_to(repo_dir))


## Executar um modelo

Selecione **exatamente um** modelo. No Mac, deixe só o Qwen ativo.


In [ ]:
executar_qwen = True
executar_ministral = False  # pesado neste Mac

selecionados = [
    nome for nome, ativo in {
        'qwen3-1.7b': executar_qwen,
        'ministral3-3b': executar_ministral,
    }.items() if ativo
]
assert len(selecionados) == 1, 'Selecione exatamente um modelo.'

cmd = [
    sys.executable, 'scripts/run_baseline.py',
    '--config', str(benchmark_config),
    '--model', selecionados[0],
]
if not HAS_CUDA:
    cmd.append('--no-4bit')

print('Comando:', ' '.join(cmd))
subprocess.run(cmd, check=True, env=os.environ.copy(), cwd=repo_dir)


In [ ]:
cfg = yaml.safe_load(benchmark_config.read_text(encoding='utf-8'))
results_dir = repo_dir / cfg['benchmark']['output_dir']
assert results_dir.exists(), f'Sem resultados em {results_dir}'

for result in sorted(results_dir.glob('*.jsonl')):
    completed = sum(1 for line in result.open(encoding='utf-8') if line.strip())
    print(f'{result.stem}: {completed} respostas → {result}')

out_csv = results_dir / 'model_comparison.csv'
subprocess.run(
    [
        sys.executable, 'scripts/compare_models.py',
        '--results', str(results_dir),
        '--output', str(out_csv),
    ],
    check=True, cwd=repo_dir,
)

import pandas as pd
df = pd.read_csv(out_csv)
try:
    display(df)
except NameError:
    print(df.to_string(index=False))


## Empacotar resultados


In [ ]:
out_root = repo_dir / 'outputs'
out_root.mkdir(parents=True, exist_ok=True)
output_archive = shutil.make_archive(
    str(out_root / 'celx-benchmark-results'),
    'zip',
    root_dir=results_dir,
)
print('Resultados:', output_archive)
print('Avalie com docs/RUBRICA.md e registre em docs/DECISOES.md.')
print('Próximo: notebooks/02_treino_qlora.ipynb (após SFT).')
